In [0]:
# 1 - Criação do Database da camada Silver:

"""
Descrição geral: Import de todos os módulos necessários para as transformações desta parte do notebook de uma vez, no topo, garantindo que não aconteçam erros de importação no meio da execução e facilitando a leitura.

Novamente, usamos IF NOT EXISTS para garantir a idempotência, ou seja, evitar problemas caso o Schema já exista.

pyspark.sql.functions (F):
- Funções nativas distribuídas do Spark (col, when, lit, datediff, upper, coalesce, current_timestamp, try_to_timestamp, etc.).
- Rodam nos workers do cluster — muito mais eficientes que UDFs Python.
  
pyspark.sql.window.Window:
- Define janelas de particionamento e ordenação para Window Functions.
- Usado na deduplicação sênior de dim_consumidores.
  
pyspark.sql.types:
- Tipos explícitos para cast() — StringType, IntegerType, DoubleType, TimestampType, DateType.
"""

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StringType, IntegerType, DoubleType, TimestampType, DateType)

spark.sql("""
    CREATE DATABASE IF NOT EXISTS silver
    COMMENT 'Camada Silver — Dados limpos, tipados e com regras de negócio aplicadas'
""")

print("Imports concluídos.")
print("Banco de dados 'silver' criado")
print()
print("Databases disponíveis:")
spark.sql("SHOW DATABASES").show(truncate=False)

Imports concluídos.
Banco de dados 'silver' criado

Databases disponíveis:
+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|information_schema|
|olist_ecommerce   |
|silver            |
+------------------+



In [0]:
# 2 - Tratamento da tabela de consumidores:

"""
Descrição geral do passo a passo que está acontecendo na célula:

Passo 1: Leitura da bronze:
- spark.table('bronze.tb_customers') lê a tabela Delta registrada no metastore. Nenhuma modificação é feita na Bronze.
  
Passo 2: Renomeação e tipagem (.select com alias e cast):
Usamos .select() em vez de múltiplos .withColumnRenamed() porque:
- É mais legível quando há muitas colunas para renomear.
- Permite aplicar cast() inline, junto com o alias.
- Seleciona apenas as colunas necessárias e descarta o restante.
CEP como StringType:
- Na Bronze, inferSchema pode ter inferido o CEP como IntegerType.
- CEP '01310' viraria 1310 (inteiro) — perdendo o zero à esquerda.
- cast(StringType()) garante que o CEP seja sempre tratado como texto.
  
Passo 3: 
Upper Case (F.upper):
- Padroniza cidade e estado em maiúsculas.
- Evita duplicatas semânticas como 'São Paulo' vs 'SAO PAULO'.
- F.upper() é uma função nativa do Spark, ou seja, roda no cluster.
  
Passo 4 - Deduplicação Sênior com Window Function:
  
Window.partitionBy('id_consumidor'):
- Cria uma "janela" separada para cada id_consumidor distinto.
- Dentro de cada janela, os registros são ordenados. 

.orderBy(F.col('timestamp_ingestion').desc()):
- Ordena do mais recente para o mais antigo dentro de cada janela.
- O registro mais recente fica na posição 1.

F.row_number().over(janela_dedup):
- Atribui o número 1 ao registro mais recente de cada consumidor.
- Registros duplicados recebem números 2, 3, 4...
  
Passo 5: Filtro e Drop da coluna auxiliar:
- Mantém apenas o registro com rn=1 (o mais recente por consumidor).
- .drop('rn'): remove a coluna auxiliar, já que não faz parte do modelo.
"""

# Definição da janela de deduplicação: 

# A partição ocorre por id_consumidor e a ordenação ocorre pelo timestamp mais recente
janela_dedup_consumidor = (
    Window
        .partitionBy("id_consumidor")
        .orderBy(F.col("timestamp_ingestion").desc())
)

# Leitura, transformação e gravação:
df_consumidores = (
    spark.table("bronze.tb_customers")

    # Renomeação de colunas para português com tipagem explícita:
    .select(
        F.col("customer_id").cast(StringType()).alias("id_consumidor"),
        F.col("customer_zip_code_prefix").cast(StringType()).alias("prefixo_cep"),
        F.col("customer_city").cast(StringType()).alias("cidade"),
        F.col("customer_state").cast(StringType()).alias("estado"),
        F.col("customer_name").cast(StringType()).alias("nome_consumidor"),
        F.col("timestamp_ingestion"),
    )

    # Upper Case em cidade e estado:
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))

    # Deduplicação Sênior: row_number dentro de cada partição de id_consumidor
    .withColumn("rn", F.row_number().over(janela_dedup_consumidor))
    .filter(F.col("rn") == 1)
    .drop("rn")  # Coluna auxiliar — removida após o filtro
)

# Gravação na Silver
(
    df_consumidores.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_consumidores")
)

total = spark.table("silver.dim_consumidores").count()
print(f"silver.dim_consumidores criada: {total:,} registros únicos")
print()
print("Amostra:")
display(spark.table("silver.dim_consumidores").limit(5))

silver.dim_consumidores criada: 99,441 registros únicos

Amostra:


id_consumidor,prefixo_cep,cidade,estado,nome_consumidor,timestamp_ingestion
00012a2ce6f8dcda20d059ce98491703,6273,OSASCO,SP,Dr. Davi Pinto,2026-04-04T00:30:40.967Z
000161a058600d5901f007fab4c27140,35550,ITAPECERICA,MG,Sr. Ravi Lucca Sousa,2026-04-04T00:30:40.967Z
0001fd6190edaaf884bcaf3d49edf079,29830,NOVA VENECIA,ES,Giovanna Ramos,2026-04-04T00:30:40.967Z
0002414f95344307404f0ace7a26f1d5,39664,MENDONCA,MG,Lívia Nogueira,2026-04-04T00:30:40.967Z
000379cdec625522490c315e70c7a9fb,4841,SAO PAULO,SP,Srta. Maria Laura Moura,2026-04-04T00:30:40.967Z


In [0]:
# 3 - Tratamento da tabela de pedidos:

#FAZER VERIFICAÇÃO POSTERIOR DO ARQUIVO ORIGINAL COM A VERSÃO CORRIGIDA

""" 
Descrição geral do tratamento aplicado:

Tradução de Status com F.when() encadeado:
- F.when(condição, valor).when(condição, valor).otherwise(valor)
- Equivale a um CASE WHEN ... THEN ... END do SQL.
- .otherwise(F.col('order_status')): caso o valor não constar no mapeamento,
mantém o valor original em inglês — tolerância a novos status futuros.
  
Colunas Derivadas de Tempo com F.datediff():
- F.datediff(data_fim, data_inicio)
- Retorna a diferença em dias entre duas colunas de data/timestamp.
- Retorna null se qualquer um dos lados for null (ex: pedido não entregue).
  
tempo_entrega_dias:
- Diferença entre a entrega real e a data de compra.
- Null quando o pedido ainda não foi entregue (sem data_entrega_cliente)
  
tempo_entrega_estimado_dias:
- Diferença entre a previsão de entrega e a data de compra.
- Sempre preenchido — a data estimada é definida no momento do pedido.
  
diferenca_entrega_dias:
- tempo_entrega_dias - tempo_entrega_estimado_dias.
- Positivo = atraso. Negativo = entrega antecipada. Zero = no prazo exato.
- Null quando tempo_entrega_dias é null (pedido não entregue).
  
entrega_no_prazo (indicador textual):
- 'Não Entregue': status não é 'entregue' (pedidos cancelados, em trânsito, etc.)
- 'Sim': entregue E data real <= data estimada
- 'Não': entregue E data real >  data estimada (atraso)
- A condição de status vem ANTES da comparação de datas para evitar
que pedidos não entregues com datas nulas gerem 'Não' erroneamente.
"""

df_pedidos = (
    spark.table("bronze.tb_orders")

    # Renomeação e Tipagem explícita
    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("customer_id").cast(StringType()).alias("id_consumidor"),
        F.col("order_status").cast(StringType()).alias("status_original"),
        F.col("order_purchase_timestamp").cast(TimestampType()).alias("data_pedido"),
        F.col("order_approved_at").cast(TimestampType()).alias("data_aprovacao"),
        F.col("order_delivered_carrier_date").cast(TimestampType()).alias("data_entrega_transportadora"),
        F.col("order_delivered_customer_date").cast(TimestampType()).alias("data_entrega_cliente"),
        F.col("order_estimated_delivery_date").cast(TimestampType()).alias("data_estimada_entrega"),
        F.col("timestamp_ingestion"),
    )

    # Tradução do Status
    .withColumn(
        "status",
        F.when(F.col("status_original") == "delivered",   "entregue")
         .when(F.col("status_original") == "canceled",    "cancelado")
         .when(F.col("status_original") == "shipped",     "enviado")
         .when(F.col("status_original") == "processing",  "processando")
         .when(F.col("status_original") == "invoiced",    "faturado")
         .when(F.col("status_original") == "unavailable", "indisponível")
         .when(F.col("status_original") == "created",     "criado")
         .when(F.col("status_original") == "approved",    "aprovado")
         .otherwise(F.col("status_original"))
    )
    .drop("status_original")

    # Coluna derivada 1: Diferença de dias entre a data de compra e a de entrega
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(
            F.col("data_entrega_cliente").cast(DateType()),
            F.col("data_pedido").cast(DateType())
        )
    )

    # Coluna derivada 2: Diferença de dias entre a compra e a data estimada para a entrega
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(
            F.col("data_estimada_entrega").cast(DateType()),
            F.col("data_pedido").cast(DateType())
        )
    )

    # Coluna derivada 3: Diferença entre o tempo real e o estimado para a entrega
    # Positivo = Atraso / Negativo = Antecipado / Null = Não Entregue
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )

    # Coluna derivada 4: Indicador de Pontualidade
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("status") != "entregue", "Não Entregue")
         .when(
             F.col("data_entrega_cliente") <= F.col("data_estimada_entrega"),
             "Sim"
         )
         .otherwise("Não")
    )
)

# Gravação na Silver:
(
    df_pedidos.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.fat_pedidos")
)

total = spark.table("silver.fat_pedidos").count()
print(f"silver.fat_pedidos criada: {total:,} registros")
print()
print("Distribuição de status após tradução:")
display(
    spark.table("silver.fat_pedidos")
        .groupBy("status")
        .count()
        .orderBy(F.col("count").desc())
)
print()
print("Amostra com colunas derivadas:")
display(
    spark.table("silver.fat_pedidos")
        .select(
            "id_pedido", "status", "tempo_entrega_dias",
            "tempo_entrega_estimado_dias", "diferenca_entrega_dias", "entrega_no_prazo"
        )
        .limit(5)
)

silver.fat_pedidos criada: 99,441 registros

Distribuição de status após tradução:


status,count
entregue,96478
enviado,1107
cancelado,625
indisponível,609
faturado,314
processando,301
criado,5
aprovado,2



Amostra com colunas derivadas:


id_pedido,status,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,entregue,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,entregue,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,entregue,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,entregue,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,entregue,3,13,-10,Sim


In [0]:
# 4 - Tratamento da tabela de itens pedidos:

"""
Descrição geral do tratamento aplicado:

Esta tabela é relativamente simples — apenas renomeação e tipagem. Porém, 
dois pontos, em especial, merecem mais atenção:

preco_BRL e preco_frete como DoubleType:
- Na Bronze, valores monetários podem ter sido inferidos como
StringType se houver vírgulas ou formato não padrão.
- O cast(DoubleType()) garante que a Gold Layer possa somar receitas
sem erros de tipo.

shipping_limit_date como TimestampType:
- Data limite de envio — relevante para análises de SLA do vendedor.
- Mantida para uso analítico futuro na Gold Layer.

Não há deduplicação aqui:
- Um pedido pode ter múltiplos itens (order_item_id = 1, 2, 3...).
- A chave composta (id_pedido, id_item) é única — não há duplicatas
legítimas nesta tabela.
"""


df_itens = (
    spark.table("bronze.tb_order_items")

    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("order_item_id").cast(IntegerType()).alias("id_item"),
        F.col("product_id").cast(StringType()).alias("id_produto"),
        F.col("seller_id").cast(StringType()).alias("id_vendedor"),
        F.col("shipping_limit_date").cast(TimestampType()).alias("data_limite_envio"),
        F.col("price").cast(DoubleType()).alias("preco_BRL"),
        F.col("freight_value").cast(DoubleType()).alias("preco_frete"),
        F.col("timestamp_ingestion"),
    )
)

(
    df_itens.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.fat_itens_pedidos")
)

total = spark.table("silver.fat_itens_pedidos").count()
print(f"silver.fat_itens_pedidos criada: {total:,} registros")
print()
print("Schema da tabela:")
spark.table("silver.fat_itens_pedidos").printSchema()
print()
print("Amostra:")
display(spark.table("silver.fat_itens_pedidos").limit(5))

silver.fat_itens_pedidos criada: 112,650 registros

Schema da tabela:
root
 |-- id_pedido: string (nullable = true)
 |-- id_item: integer (nullable = true)
 |-- id_produto: string (nullable = true)
 |-- id_vendedor: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco_BRL: double (nullable = true)
 |-- preco_frete: double (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)


Amostra:


id_pedido,id_item,id_produto,id_vendedor,data_limite_envio,preco_BRL,preco_frete,timestamp_ingestion
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-04-04T00:30:59.977Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-04-04T00:30:59.977Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-04-04T00:30:59.977Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-04-04T00:30:59.977Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-04-04T00:30:59.977Z


In [0]:
# 5 - Tratamento da tabela de pagamentos:

#FAZER VERIFICAÇÃO POSTERIOR DO ARQUIVO ORIGINAL COM A VERSÃO CORRIGIDA

"""
Descrição geral do tratamento aplicado:

Tradução do tipo de pagamento:
- Mesmo padrão usado na tabela 'fat_pedidos' para tradução de Status.
- .otherwise(F.col('payment_type')): Mantém o valor original caso surja 
um tipo não mapeado, ou seja, há tolerância para outras formas de pagamento.

Um pedido pode ter múltiplos pagamentos (ex: Boleto + Cartão de Crédito):
- payment_sequential (id_sequencia_pagamento) identifica a ordem
dos pagamentos dentro de um mesmo pedido.
- Não há deduplicação — a chave composta (id_pedido, id_sequencia) já é única.

payment_installments (Parcelas):
- Número de parcelas escolhidas pelo consumidor.
- IntegerType para evitar casos de 'meia' parcela.

payment_value (valor_pagamento):
- DoubleType — valor monetário com casas decimais.
- O arredondamento para 2 casas decimais será feito na Gold Layer para
serem apresentados.

"""

# Inglês -> Português
traducao_pagamento = (
    F.when(F.col("tipo_pagamento_original") == "credit_card",  "Cartão de Crédito")
     .when(F.col("tipo_pagamento_original") == "boleto",       "Boleto")
     .when(F.col("tipo_pagamento_original") == "voucher",      "Voucher")
     .when(F.col("tipo_pagamento_original") == "debit_card",   "Cartão de Débito")
     .when(F.col("tipo_pagamento_original") == "not_defined",  "Não Definido")
     .otherwise(F.col("tipo_pagamento_original"))  # Tolerância outros tipos de pagamento
)

df_pagamentos = (
    spark.table("bronze.tb_order_payments")

    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("payment_sequential").cast(IntegerType()).alias("id_sequencia_pagamento"),
        F.col("payment_type").cast(StringType()).alias("tipo_pagamento_original"),
        F.col("payment_installments").cast(IntegerType()).alias("parcelas"),
        F.col("payment_value").cast(DoubleType()).alias("valor_pagamento"),
        F.col("timestamp_ingestion"),
    )

    # Tradução do tipo de pagamento
    .withColumn("tipo_pagamento", traducao_pagamento)
    .drop("tipo_pagamento_original")
)

(
    df_pagamentos.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.fat_pagamentos_pedidos")
)

total = spark.table("silver.fat_pagamentos_pedidos").count()
print(f"silver.fat_pagamentos_pedidos criada: {total:,} registros")
print()
print("Distribuição por tipo de pagamento:")
display(
    spark.table("silver.fat_pagamentos_pedidos")
        .groupBy("tipo_pagamento")
        .count()
        .orderBy(F.col("count").desc())
)
print()
print("Amostra:")
display(spark.table("silver.fat_pagamentos_pedidos").limit(5))

silver.fat_pagamentos_pedidos criada: 103,886 registros

Distribuição por tipo de pagamento:


tipo_pagamento,count
Cartão de Crédito,76795
Boleto,19784
Voucher,5775
Cartão de Débito,1529
Não Definido,3



Amostra:


id_pedido,id_sequencia_pagamento,parcelas,valor_pagamento,timestamp_ingestion,tipo_pagamento
b81ef226f3fe1789b1e8b2acac839d17,1,8,99.33,2026-04-04T00:31:05.266Z,Cartão de Crédito
a9810da82917af2d9aefd1278f1dcfa0,1,1,24.39,2026-04-04T00:31:05.266Z,Cartão de Crédito
25e8ea4e93396b6fa0d3dd708e76c1bd,1,1,65.71,2026-04-04T00:31:05.266Z,Cartão de Crédito
ba78997921bbcdc1373bb41e913ab953,1,8,107.78,2026-04-04T00:31:05.266Z,Cartão de Crédito
42fdf880ba16b47b59251dd489d4441a,1,2,128.45,2026-04-04T00:31:05.266Z,Cartão de Crédito


In [0]:
# 6 - Tratamento da tabela de avaliações:

#FAZER VERIFICAÇÃO POSTERIOR DO ARQUIVO ORIGINAL COM A VERSÃO CORRIGIDA

"""
Descrição geral do tratamento aplicado:

Passo 1 - Conversão tolerante a falhas com try_to_timestamp:
- F.try_to_timestamp(coluna, formato): Tenta converter a string para timestamp usando o formato informado.
- Se a conversão falhar (valor nulo, string inválida, formato diferente), retorna NULL em vez de lançar uma exceção e parar o job.
- O formato 'yyyy-MM-dd HH:mm:ss' cobre o padrão do dataset.

Passo 2 - Filtros de qualidade (Remoção de registros inválidos):

. Filtro 1 — Pedido inválido:
- Remove registros onde order_id é nulo ou string vazia.
- Uma avaliação sem pedido é impossível de associar ao modelo e causaria problemas nos joins da Gold.
- F.trim() remove espaços em branco antes de checar o tamanho.

. Filtro 2 — Datas no futuro:
- Remove registros onde a data de criação da avaliação é maior que o timestamp atual (F.current_timestamp()).
- Datas no futuro indicam erro de sistema
- Usamos F.coalesce(data, F.lit('2000-01-01')): se a data for null após o try_to_timestamp, assumimos uma data antiga (passada) para
que o registro NÃO seja filtrado.
- Registros com data inválida (null) são mantidos — apenas datas explicitamente no futuro são removidas.

Passo 3 - Tratamento de nulos com F.coalesce():
. F.coalesce(col_a, col_b):
- Retorna o primeiro valor não nulo entre os argumentos.
. F.coalesce(F.col('review_comment_title'), F.lit('Sem título')):
- Se o título for null, substitui por 'Sem título'.
- F.trim() antes garante que strings com apenas espaços também sejam tratadas: trim(' ') → '' → nullif → null → coalesce → padrão.

Observação: 
A especificação mapeia review_comment_title e review_comment_message para o mesmo nome 'comentario_avaliacao', o que resultaria em perda de dados.
Optei por por preservar ambas as colunas com nomes distintos (titulo_avaliacao e comentario_avaliacao), mantendo a semântica de cada campo.
"""

df_avaliacoes = (
    spark.table("bronze.tb_order_reviews")

    # Passo 1: Conversão tolerante a falhas com try_to_timestamp
    .withColumn(
        "data_criacao_avaliacao",
        F.try_to_timestamp(F.col("review_creation_date"), F.lit("yyyy-MM-dd HH:mm:ss"))
    )
    .withColumn(
        "data_resposta_avaliacao",
        F.try_to_timestamp(F.col("review_answer_timestamp"), F.lit("yyyy-MM-dd HH:mm:ss"))
    )

    # Passo 2: Remove avaliações com pedido inválido (nulo ou vazio)
    .filter(
        F.col("order_id").isNotNull() &
        (F.length(F.trim(F.col("order_id"))) > 0)
    )

    # Passo 2.1: Remove datas de criação explicitamente no futuro
    # Registros com data nula são mantidos (coalesce para data antiga)
    .filter(
        F.coalesce(
            F.col("data_criacao_avaliacao"),
            F.lit("2000-01-01").cast(TimestampType())
        ) <= F.current_timestamp()
    )

    # Passo 3: Tratamento de nulos em campos de texto
    .withColumn(
        "titulo_avaliacao",
        F.coalesce(
            F.nullif(F.trim(F.col("review_comment_title")), F.lit("")), # Transformação de string vazia em null
            F.lit("Sem título")
        )
    )
    .withColumn(
        "comentario_avaliacao",
        F.coalesce(
            F.nullif(F.trim(F.col("review_comment_message")), F.lit("")),
            F.lit("Sem comentário")
        )
    )

    # Passo 4: Seleção final com Nomes em Português e Tipagem correta
    .select(
        F.col("review_id").cast(StringType()).alias("id_avaliacao"),
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("review_score").cast(StringType()).alias("nota_avaliacao"),
        F.col("titulo_avaliacao"),
        F.col("comentario_avaliacao"),
        F.col("data_criacao_avaliacao"),
        F.col("data_resposta_avaliacao"),
        F.col("timestamp_ingestion"),
    )
)

(
    df_avaliacoes.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.fat_avaliacoes_pedidos")
)

total_bronze = spark.table("bronze.tb_order_reviews").count()
total_silver = spark.table("silver.fat_avaliacoes_pedidos").count()
removidos    = total_bronze - total_silver

print(f"Registros na Bronze: {total_bronze:,}")
print(f"Registros na Silver: {total_silver:,}")
print(f"Removidos pelos filtros: {removidos:,}")
print()
print("Distribuição de notas:")
display(
    spark.table("silver.fat_avaliacoes_pedidos")
        .groupBy("nota_avaliacao")
        .count()
        .orderBy("nota_avaliacao")
)
print()
print("Amostra:")
display(spark.table("silver.fat_avaliacoes_pedidos").limit(5))

Registros na Bronze: 104,162
Registros na Silver: 101,922
Removidos pelos filtros: 2,240

Distribuição de notas:


nota_avaliacao,count
null,140
"""",1
"BARALHO KEM COM NAIPES GRANDES E VEIRO COM NAIPES PEQUENOS. COMO FAÇO PARA DEVOLVER? OU O NÚMERO DO TELEFONE DO OUVIDOR?""",1
"Desta vez nã""",1
M,1
NAO confere para o modelo anunciado,1
"VOU RECEBER DEPOIS A OUTRA UNIDADE???""",1
"a URA não resolve nada.""",1
"a previsão dada pela Loja era de até 27/12/2017.""",1
"a pulseira do mesmo já estava com havaria e em seguida quando coloquei o horário correto em alguns minutos o relógio parou e não voltou a funcionar""",1



Amostra:


id_avaliacao,id_pedido,nota_avaliacao,titulo_avaliacao,comentario_avaliacao,data_criacao_avaliacao,data_resposta_avaliacao,timestamp_ingestion
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,Sem título,Sem comentário,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,2026-04-04T00:31:10.621Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,Sem título,Sem comentário,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,2026-04-04T00:31:10.621Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,Sem título,Sem comentário,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,2026-04-04T00:31:10.621Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Sem título,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,2026-04-04T00:31:10.621Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Sem título,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,2026-04-04T00:31:10.621Z


In [0]:
# 7 - Tratamento da tabela de produtos:

"""
Descrição geral do tratamento aplicado:

Deduplicação Sênior com Window Function:
- Mesma lógica aplicada na tabela 'dim_consumidores'.
- Particiona-se por id_produto e se ordena pelo timestamp_ingestion de forma decrescente, garantindo que apenas o registro mais recente de cada produto seja mantido.
- row_number() == 1 seleciona o registro mais recente de cada partição.

Tipagem das dimensões físicas como DoubleType:
- Peso, comprimento, altura e largura são medidas contínuas que podem ter casos com casas decimais.
- DoubleType é o tipo correto para evitar truncamento de precisão.

Tipagem de contagens como IntegerType:
- 'quantidade_fotos' e os tamanhos de 'nome/descrição' são valores inteiros.
- Embora o dataset os armazene como Double, o cast para IntegerType representa melhor a semântica desses campos.

Mapeamento de colunas conforme especificação:
- product_id -> id_produto
- product_name -> nome_produto
- product_category_name -> categoria_produto
- product_weight_g -> peso_produto_gramas
- product_length_cm -> comprimento_centimetros
- product_height_cm -> altura_centimetros
- product_width_cm -> largura_centimetros
- product_photos_qty -> quantidade_fotos
- product_name_lenght -> tamanho_nome_produto
- product_description_lenght -> tamanho_descricao_produto
"""

# Janela de deduplicação por produto:
janela_dedup_produto = (
    Window
        .partitionBy("id_produto")
        .orderBy(F.col("timestamp_ingestion").desc())
)

df_produtos = (
    spark.table("bronze.tb_products")

    # Renomeação e Tipagem explícita de todas as colunas:
    .select(
        F.col("product_id").cast(StringType()).alias("id_produto"),
        F.col("product_name").cast(StringType()).alias("nome_produto"),
        F.col("product_category_name").cast(StringType()).alias("categoria_produto"),
        F.col("product_weight_g").cast(DoubleType()).alias("peso_produto_gramas"),
        F.col("product_length_cm").cast(DoubleType()).alias("comprimento_centimetros"),
        F.col("product_height_cm").cast(DoubleType()).alias("altura_centimetros"),
        F.col("product_width_cm").cast(DoubleType()).alias("largura_centimetros"),
        F.col("product_photos_qty").cast(IntegerType()).alias("quantidade_fotos"),
        F.col("product_name_lenght").cast(IntegerType()).alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast(IntegerType()).alias("tamanho_descricao_produto"),
        F.col("timestamp_ingestion"),
    )

    # Deduplicação Sênior: Mantém apenas o registro mais recente por produto
    .withColumn("rn", F.row_number().over(janela_dedup_produto))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

(
    df_produtos.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_produtos")
)

total = spark.table("silver.dim_produtos").count()
print(f"silver.dim_produtos criada: {total:,} registros únicos")
print()
print("Schema da tabela:")
spark.table("silver.dim_produtos").printSchema()
print()
print("Amostra:")
display(spark.table("silver.dim_produtos").limit(5))

silver.dim_produtos criada: 32,951 registros únicos

Schema da tabela:
root
 |-- id_produto: string (nullable = true)
 |-- nome_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = true)
 |-- peso_produto_gramas: double (nullable = true)
 |-- comprimento_centimetros: double (nullable = true)
 |-- altura_centimetros: double (nullable = true)
 |-- largura_centimetros: double (nullable = true)
 |-- quantidade_fotos: integer (nullable = true)
 |-- tamanho_nome_produto: integer (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)


Amostra:


id_produto,nome_produto,categoria_produto,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,quantidade_fotos,tamanho_nome_produto,tamanho_descricao_produto,timestamp_ingestion
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,perfumaria,300.0,20.0,16.0,16.0,6,53,596,2026-04-04T00:31:21.594Z
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,automotivo,1225.0,55.0,10.0,26.0,4,56,752,2026-04-04T00:31:21.594Z
0009406fd7479715e4bef61dd91f2462,Toalha de Banho Premium,cama_mesa_banho,300.0,45.0,15.0,35.0,2,50,266,2026-04-04T00:31:21.594Z
000b8f95fcb9e0096488278317764d19,Tábua de Corte,utilidades_domesticas,550.0,19.0,24.0,12.0,3,25,364,2026-04-04T00:31:21.594Z
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,relogios_presentes,250.0,22.0,11.0,15.0,4,48,613,2026-04-04T00:31:21.594Z


In [0]:
# 8 - Tratamento da tabela de vendedores:

"""
Descrição geral do tratamento aplicado:

Deduplicação Sênior com Window Function:
- Particiona-se por 'id_vendedor' e se ordena pelo 'timestamp_ingestion' de forma decrescente.
- Garante que cada seller_id apareça apenas uma vez na dimensão, sempre com os dados mais recentes.

Upper Case em cidade e estado:
- Mesma regra aplicada em 'dim_consumidores'.
- Evita duplicatas semânticas por diferença de capitalização.

CEP como StringType:
- Mesmo motivo de 'dim_consumidores': zeros à esquerda seriam perdidos se o CEP fosse tratado como IntegerType.

Mapeamento de colunas conforme especificação:
- seller_id -> id_vendedor
- seller_name -> nome_vendedor
- seller_zip_code_prefix -> prefixo_cep
- seller_city -> cidade
- seller_state -> estado

Observação: 
'seller_registration_date' existe no novo dataset, mas não consta na documentação enviada da task, ou seja, 
foi omitido de forma intencional.
"""

# Janela de Deduplicação por vendedor:
janela_dedup_vendedor = (
    Window
        .partitionBy("id_vendedor")
        .orderBy(F.col("timestamp_ingestion").desc())
)

df_vendedores = (
    spark.table("bronze.tb_sellers")

    # Renomeação e Tipagem explícita:
    .select(
        F.col("seller_id").cast(StringType()).alias("id_vendedor"),
        F.col("seller_name").cast(StringType()).alias("nome_vendedor"),
        F.col("seller_zip_code_prefix").cast(StringType()).alias("prefixo_cep"),
        F.col("seller_city").cast(StringType()).alias("cidade"),
        F.col("seller_state").cast(StringType()).alias("estado"),
        F.col("timestamp_ingestion"),
    )

    # Upper Case em Cidade e Estado:
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))

    # Deduplicação Sênior: Mantém apenas o registro mais recente por vendedor
    .withColumn("rn", F.row_number().over(janela_dedup_vendedor))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

(
    df_vendedores.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_vendedores")
)

total = spark.table("silver.dim_vendedores").count()
print(f"silver.dim_vendedores criada: {total:,} registros únicos")
print()
print("Amostra:")
display(spark.table("silver.dim_vendedores").limit(5))

silver.dim_vendedores criada: 3,095 registros únicos

Amostra:


id_vendedor,nome_vendedor,prefixo_cep,cidade,estado,timestamp_ingestion
0015a82c2db000af6aaaf3ae2ecb0532,Amanda Sá,9080,SANTO ANDRE,SP,2026-04-04T00:31:26.758Z
001cca7ae9ae17fb1caed9dfb1094831,Vinicius Nogueira,29156,CARIACICA,ES,2026-04-04T00:31:26.758Z
001e6ad469a905060d959994f1b41e4f,Ana Clara Moreira,24754,SAO GONCALO,RJ,2026-04-04T00:31:26.758Z
002100f778ceb8431b7a1020ff7ab48f,Srta. Emanuella Rezende,14405,FRANCA,SP,2026-04-04T00:31:26.758Z
003554e2dce176b5555353e4f3555ac8,Rebeca Costa,74565,GOIANIA,GO,2026-04-04T00:31:26.758Z


In [0]:
# 9 - Tratamento da tabela de tradução de categorias:

"""
Descrição geral do tratamento aplicado:

Esta tabela é um dicionário de tradução de categorias de produtos.

A transformação é apenas de renomeação, conforme especificação:
- 'product_category_name' -> 'nome_produto_pt' (Nome em Português)
- 'product_category_name_english' -> 'nome_produto_en' (Nome em Inglês)

Não há deduplicação:
- A tabela tem 71 categorias únicas, sendo uma linha por categoria.
- Cada 'product_category_name' é naturalmente único.
"""

df_categorias = (
    spark.table("bronze.tb_product_category_name_translation")

    .select(
        F.col("product_category_name").cast(StringType()).alias("nome_produto_pt"),
        F.col("product_category_name_english").cast(StringType()).alias("nome_produto_en"),
        F.col("timestamp_ingestion"),
    )
)

(
    df_categorias.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_categoria_produtos_traducao")
)

total = spark.table("silver.dim_categoria_produtos_traducao").count()
print(f"silver.dim_categoria_produtos_traducao criada: {total:,} categorias")
print()
print("Amostra:")
display(spark.table("silver.dim_categoria_produtos_traducao").limit(10))

silver.dim_categoria_produtos_traducao criada: 71 categorias

Amostra:


nome_produto_pt,nome_produto_en,timestamp_ingestion
beleza_saude,health_beauty,2026-04-04T00:31:30.764Z
informatica_acessorios,computers_accessories,2026-04-04T00:31:30.764Z
automotivo,auto,2026-04-04T00:31:30.764Z
cama_mesa_banho,bed_bath_table,2026-04-04T00:31:30.764Z
moveis_decoracao,furniture_decor,2026-04-04T00:31:30.764Z
esporte_lazer,sports_leisure,2026-04-04T00:31:30.764Z
perfumaria,perfumery,2026-04-04T00:31:30.764Z
utilidades_domesticas,housewares,2026-04-04T00:31:30.764Z
telefonia,telephony,2026-04-04T00:31:30.764Z
relogios_presentes,watches_gifts,2026-04-04T00:31:30.764Z


In [0]:
# 10 - Tratamento da tabela de cotação do dólar:

"""
Descrição geral do tratamento aplicado:

Passo 1 — Preparação das cotações brutas:
- A tabela 'bronze.tb_cotacao_dolar' é lida e se extrai apenas a data (sem hora) da coluna 'dataHoraCotacao',
pois é desejada apenas uma cotação por dia.
- Caso haja múltiplas cotações no mesmo dia, será considerada a última cotação do dia com uma agregação max() 
na hora — representando o fechamento do dia, que é a cotação de referência para fins de semana.

Passo 2 — Geração do Calendário Contínuo:
- F.sequence(data_inicio, data_fim, F.expr('INTERVAL 1 DAY')):
Gera um array com todas as datas do período, um dia por vez.
- F.explode() transforma esse array em linhas individuais.
- O resultado é um DataFrame com uma linha para cada dia do período, incluindo fins de semana e feriados.

Passo 3 — Left join Calendário x Cotações:
- O left join garante que todos os dias do calendário apareçam, mesmo que não haja cotação para aquele dia.
- Dias úteis terão 'cotacaoCompra' preenchido.
- Fins de semana e feriados terão 'cotacaoCompra = null'.

Passo 4 — Forward fill com last(ignorenulls=True):
- Window.orderBy('data').rowsBetween(Window.unboundedPreceding, 0):
Define uma janela que vai desde o início até a linha atual (inclusive). Isso significa que para cada linha, a janela contém todos os dias
anteriores mais o dia atual.
- F.last('cotacaoCompra', ignorenulls=True).over(janela_forward_fill):
Retorna o último valor não nulo dentro da janela. 
. Para um sábado (cotação null), a janela vai até sexta e o last(ignorenulls=True) retorna a cotação de sexta-feira. 
. Para um domingo, a janela vai até sexta também.
. Para um dia útil, retorna a cotação do próprio dia.
"""

# Passo 1 - Prepara Cotações Brutas (Uma cotação por dia):
df_cotacao_bruta = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn(
        "data",
        # Extrai apenas a data, desconsiderando a hora do campo 'dataHoraCotacao'
        F.to_date(F.col("dataHoraCotacao"))
    )
    # Caso haja múltiplas cotações no mesmo dia, mantém a última, ou seja, o fechamento
    .groupBy("data")
    .agg(F.max("cotacaoCompra").cast(DoubleType()).alias("cotacao_compra"))
)

# Passo 2 - Gera o Calendário Contínuo cobrindo todo o período das cotações:
datas_limite = df_cotacao_bruta.agg(
    F.min("data").alias("data_inicio"),
    F.max("data").alias("data_fim")
).collect()[0]

df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(datas_limite["data_inicio"]),
                F.lit(datas_limite["data_fim"]),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data")
    )
)

# Passo 3 - Left join (Todos os dias do calendário + cotação quando disponível):
# Fins de semana e feriados ficam com cotacao_compra = null após o join
df_cotacao_com_gaps = (
    df_calendario
    .join(df_cotacao_bruta, on="data", how="left")
    .orderBy("data")
)

# Passo 4 - Forward fill (Propaga a última cotação conhecida para dias sem cotação):
# A janela cobre desde o início até a linha atual (rowsBetween)
# last(ignorenulls=True) retorna o último valor não nulo na janela
janela_forward_fill = (
    Window
        .orderBy("data")
        .rowsBetween(Window.unboundedPreceding, 0)
)

df_cotacao_final = (
    df_cotacao_com_gaps
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(janela_forward_fill)
    )
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

(
    df_cotacao_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.dim_cotacao_dolar")
)

total = spark.table("silver.dim_cotacao_dolar").count()
total_bruta = df_cotacao_bruta.count()
print(f"Dias no calendário contínuo: {total:,}")
print(f"Dias com cotação real (API): {total_bruta:,}")
print(f"Dias preenchidos por fill: {total - total_bruta:,}")
print()
print("Amostra - Verificando preenchimento de fins de semana:")
display(
    spark.table("silver.dim_cotacao_dolar")
    .withColumn("dia_semana", F.date_format(F.col("data"), "EEEE"))
    .filter(F.col("dia_semana").isin("Saturday", "Sunday", "Friday"))
    .orderBy("data")
    .limit(9)
)

Dias no calendário contínuo: 1,093
Dias com cotação real (API): 750
Dias preenchidos por fill: 343

Amostra - Verificando preenchimento de fins de semana:


data,cotacao_compra,timestamp_ingestion,dia_semana
2016-01-08,4.0244,2026-04-04T20:44:18.732Z,Friday
2016-01-09,4.0244,2026-04-04T20:44:18.732Z,Saturday
2016-01-10,4.0244,2026-04-04T20:44:18.732Z,Sunday
2016-01-15,4.0396,2026-04-04T20:44:18.732Z,Friday
2016-01-16,4.0396,2026-04-04T20:44:18.732Z,Saturday
2016-01-17,4.0396,2026-04-04T20:44:18.732Z,Sunday
2016-01-22,4.1226,2026-04-04T20:44:18.732Z,Friday
2016-01-23,4.1226,2026-04-04T20:44:18.732Z,Saturday
2016-01-24,4.1226,2026-04-04T20:44:18.732Z,Sunday


In [0]:
# 11 - Construção da tabela 'fat_pedido_total':

"""
Descrição geral do tratamento aplicado:

Aqui, são consolidados os pedidos, valores pagos e a cotação do dólar em uma visão analítica única
por pedido. 

Passo 1 — Agregação de pagamentos por pedido:
- 'fat_pagamentos_pedidos' tem uma linha por forma de pagamento.
- Um pedido pode ter sido pago com Cartão + Voucher, por exemplo, gerando 2 linhas.
- Agrupa-se por 'id_pedido' e somamos 'valor_pagamento' para obter o valor total pago por pedido em BRL.

Passo 2 — Join com 'fat_pedidos':
- LEFT JOIN garante que todos os pedidos apareçam mesmo que não tenham pagamento registrado (por exemplo: 
pedidos cancelados antes do pagamento).
- São trazidos 'id_consumidor', 'status' e 'data_pedido' de 'fat_pedidos'.

Passo 3 — Join com 'dim_cotacao_dolar':
- Converte-se 'data_pedid'o para DateType para fazer o join com a coluna 'data' de 'dim_cotacao_dolar' (que é DateType).
- LEFT JOIN garante que pedidos sem cotação disponível apareçam com 'valor_total_pago_usd' = 'null' em vez de serem descartados.

Passo 4 — Conversão BRL -> USD:
- 'valor_total_pago_usd' = 'valor_total_pago_brl'/'cotacao_compra'.
- Divisão por null (pedido sem cotação) retorna null.

Passo 5 — Arredondamento obrigatório para 2 casas decimais:
- F.round(col, 2): arredonda para exatamente 2 casas decimais.

Colunas finais:
- 'id_pedido', 'id_consumidor', 'status', 'valor_total_pago_brl', 'valor_total_pago_usd', 'data_pedido'.
"""

# Passo 1 - Agregam-se os pagamentos (Soma Total por pedido):
df_pagamentos_agg = (
    spark.table("silver.fat_pagamentos_pedidos")
    .groupBy("id_pedido")
    .agg(
        F.sum("valor_pagamento").alias("valor_total_pago_brl")
    )
)

# Passo 2 - Join Pedidos + Pagamentos agregados:
df_pedidos = spark.table("silver.fat_pedidos")
df_cotacao = spark.table("silver.dim_cotacao_dolar")

df_pedido_total = (
    df_pedidos
    .select("id_pedido", "id_consumidor", "status", "data_pedido")
    .join(df_pagamentos_agg, on="id_pedido", how="left")

    # Passo 3 - Join com cotação do dólar pelo dia do pedido:
    # Conversão de data_pedido (TimestampType) para DateType
    .join(
        df_cotacao.select("data", "cotacao_compra"),
        on=F.to_date(F.col("data_pedido")) == F.col("data"),
        how="left"
    )
    .drop("data")  # Removendo a coluna auxiliar do join

    # Passo 4 - Conversão de BRL para USD usando a cotação do dia:
    .withColumn(
        "valor_total_pago_usd",
        F.col("valor_total_pago_brl") / F.col("cotacao_compra")
    )
    .drop("cotacao_compra")  # Coluna auxiliar (Não faz parte do modelo final)

    # Passo 5 - Arredondamento obrigatório para 2 casas decimais (Usa-se a função F.round()):
    .withColumn("valor_total_pago_brl", F.round(F.col("valor_total_pago_brl"), 2))
    .withColumn("valor_total_pago_usd", F.round(F.col("valor_total_pago_usd"), 2))

    # Seleciondo apenas as colunas necessárias, em ordem:
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        "data_pedido",
    )
)

(
    df_pedido_total.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("silver.fat_pedido_total")
)

total = spark.table("silver.fat_pedido_total").count()
sem_usd = spark.table("silver.fat_pedido_total").filter(F.col("valor_total_pago_usd").isNull()).count()

print(f"silver.fat_pedido_total criada: {total:,} registros")
print(f"Pedidos sem conversão USD: {sem_usd:,} (pedidos fora do período de cotação)")
print()
print("Amostra:")
display(spark.table("silver.fat_pedido_total").limit(5))
print()
print("Distribuição por status:")
display(
    spark.table("silver.fat_pedido_total")
    .groupBy("status")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(F.sum("valor_total_pago_brl"), 2).alias("receita_total_brl")
    )
    .orderBy(F.col("total_pedidos").desc())
)

silver.fat_pedido_total criada: 99,441 registros
Pedidos sem conversão USD: 1 (pedidos fora do período de cotação)

Amostra:


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,38.71,12.24,2017-10-02T10:56:33.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,141.46,37.77,2018-07-24T20:41:37.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,179.12,47.75,2018-08-08T08:38:49.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,72.2,22.02,2017-11-18T19:28:06.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,28.62,8.72,2018-02-13T21:18:39.000Z



Distribuição por status:


status,total_pedidos,receita_total_brl
entregue,96478,1.542246177E7
enviado,1107,177213.96
cancelado,625,143255.6
indisponível,609,126479.51
faturado,314,69137.99
processando,301,69394.11
criado,5,688.1
aprovado,2,241.08


In [0]:
# 12 - Otimização física das tabelas fato:

#FAZER VERIFICAÇÃO POSTERIOR DO ARQUIVO ORIGINAL COM A VERSÃO CORRIGIDA

"""
Descrição geral da otimização aplicada:

OPTIMIZE {tabela} ZORDER BY (col1, col2):
- OPTIMIZE compacta os arquivos Delta fragmentados em arquivos maiores e mais eficientes para leitura.
- ZORDER BY reorganiza os dados dentro dos arquivos para co-localizar registros com valores próximos das colunas especificadas.
- Queries com filtros em 'id_pedido' ou 'data_pedido' se beneficiam do 'data skipping': o Spark lê apenas os arquivos que podem conter
os valores buscados, ignorando os demais.

Aplicam-se nas 3 tabelas fato principais:
- 'fat_pedidos': Maior tabela e será a mais consultada.
- 'fat_pedido_total': Tabela central de receita — Joins frequentes.
- 'fat_avaliacoes_pedidos': Análises de satisfação — Filtros por data.

Não se aplicam em tabelas dimensão (dim_*) pois:
- São menores e já eficientes para leitura completa.
- Dimensões raramente são filtradas diretamente, pois costumam chegar via Join.
"""

TABELAS_FATO_OTIMIZAR = [
    ("silver.fat_pedidos", "id_pedido, data_pedido"),
    ("silver.fat_pedido_total", "id_pedido, data_pedido"),
    ("silver.fat_avaliacoes_pedidos", "id_pedido, data_criacao_avaliacao"),
]

print("Iniciando otimização física das tabelas fato...")

for tabela, colunas_zorder in TABELAS_FATO_OTIMIZAR:
    try:
        spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas_zorder})")
        print(f"    [OK] {tabela}")
        print(f"    ZORDER BY ({colunas_zorder})")
    except Exception as e:
        print(f"    [Erro] {tabela}: {str(e)[:80]}")

print("Otimização concluída.")

Iniciando otimização física das tabelas fato...
    [OK] silver.fat_pedidos
    ZORDER BY (id_pedido, data_pedido)
    [OK] silver.fat_pedido_total
    ZORDER BY (id_pedido, data_pedido)
    [OK] silver.fat_avaliacoes_pedidos
    ZORDER BY (id_pedido, data_criacao_avaliacao)
Otimização concluída.


In [0]:
# 13 - Validação completa da camada Silver:

"""
Verifica todas as 9 tabelas da Silver:
- Existência no metastore
- Contagem de registros (> 0)
- Presença de timestamp_ingestion

O relatório final consolida o estado completo da camada Silver
antes de passar para a camda Gold.
"""

from datetime import datetime

TODAS_TABELAS_SILVER = [
    "silver.dim_consumidores",
    "silver.fat_pedidos",
    "silver.fat_itens_pedidos",
    "silver.fat_pagamentos_pedidos",
    "silver.fat_avaliacoes_pedidos",
    "silver.dim_produtos",
    "silver.dim_vendedores",
    "silver.dim_categoria_produtos_traducao",
    "silver.dim_cotacao_dolar",
    "silver.fat_pedido_total",
]

print("Relatório Final — Camada Silver")
print(f"  Concluído em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print()
print(f"  {'Tabela':<45} {'Registros':>10}  {'Status':>9}")
print("  " + "-" * 68)

for tabela in TODAS_TABELAS_SILVER:
    try:
        df = spark.table(tabela)
        n  = df.count()
        st = "OK" if n > 0 else "VERIFICAR"
        print(f"  {tabela:<45} {n:>10,}  {st:>9}")
    except Exception as e:
        print(f"  {tabela:<45} {'ERRO':>10}  {'FALHOU':>9}")

print()
print("Finalizado!")

Relatório Final — Camada Silver
  Concluído em: 04/04/2026 20:45:33

  Tabela                                         Registros     Status
  --------------------------------------------------------------------
  silver.dim_consumidores                           99,441         OK
  silver.fat_pedidos                                99,441         OK
  silver.fat_itens_pedidos                         112,650         OK
  silver.fat_pagamentos_pedidos                    103,886         OK
  silver.fat_avaliacoes_pedidos                    101,922         OK
  silver.dim_produtos                               32,951         OK
  silver.dim_vendedores                              3,095         OK
  silver.dim_categoria_produtos_traducao                71         OK
  silver.dim_cotacao_dolar                           1,093         OK
  silver.fat_pedido_total                           99,441         OK

Finalizado!
